# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20, 10. отобразили индекс b и первую строку 
2. Шымкент 3, NaN, NaN series складываются по индексу, а если нет, тогда nan
3. (2, 2) условие убирает первую строку, оставшиеся строки соответствуют x > 1
4. series, dataframe одни кв скобки - series, две - df
5. [1, 2] 5 остались индексы 1 и 2, записали их в список. значение в 1 строке, 2 столбце таблицы f - 5 

In [1]:
import pandas as pd
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])

20 10


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
# 1
print('Строк и столбцов в таблице:', df.shape)
print('Типы столбцов:')
print(df.dtypes)

Строк и столбцов в таблице: (5463, 8)
Типы столбцов:
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object


In [6]:
print('Пропусков в `delivery_days`:', len(df) - df['delivery_days'].count())

Пропусков в `delivery_days`: 160


In [7]:
print('Уникальных значений столбцов:')
print(df[['city', 'category', 'status', 'customer_id']].nunique())

Уникальных значений столбцов:
city             14
category          5
status            2
customer_id    2003
dtype: int64


In [8]:
for u in ['city', 'category', 'status']:
    print(u, df[u].unique())

city <ArrowStringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str
category <ArrowStringArray>
['Одежда', 'Красота', 'Дом', 'Книги', 'Электроника']
Length: 5, dtype: str
status <ArrowStringArray>
['оплачен', 'отменён']
Length: 2, dtype: str


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 671.2 KB


Главные проблемы данных, которые уже видны на этом этапе: 
- неверный тип данных в столбце `order_date` (возникнут трудности при фильтрации данных по дате, если не исправить)
- отсутствие единого формата написания названий городов (один город может отображаться как несколько разных, искажение данных)
- необработанные пропуски в `delivery_days` (неточности во время описательной статистики данных)

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [10]:
# 2
not_test_and_paid = df[not_test & paid]
not_test_and_paid['amount'].describe()

count      4901.000000
mean      14672.391349
std       13249.340787
min         720.000000
25%        5880.000000
50%       10300.000000
75%       18690.000000
max      139040.000000
Name: amount, dtype: float64

In [11]:
print('Средний чек:', round(not_test_and_paid['amount'].mean(), 1))
print('Медиана:', not_test_and_paid['amount'].median())
print('25-й перцентиль:', not_test_and_paid['amount'].quantile(0.25))
print('75-й перцентиль:', not_test_and_paid['amount'].quantile(0.75))

Средний чек: 14672.4
Медиана: 10300.0
25-й перцентиль: 5880.0
75-й перцентиль: 18690.0


Показать в отчете стоит и среднее, и медиану. Благодаря им мы имеем первичное представление о степени "скошенности" данных. Высокий средний чек (14672) может говорить о наличии больших или аномальных значений, а медиана (10300) показывает самое серединное значение всех чеков. При скошенности распределения вправо (что вероятно при средней больше чем медиана), медиана лучше отображает положение вещей. Это и есть наш "типичный" чек - 10300.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [12]:
# 3
print('Всего заказов со статусом:', len(df['status']))
print('Отмененных заказов:', len(df[~paid]))
print(f'Доля отменённых заказов среди всех заказов: {len(df[~paid]) / len(df['status']) * 100:.1f}%')

Всего заказов со статусом: 5463
Отмененных заказов: 559
Доля отменённых заказов среди всех заказов: 10.2%


In [13]:
ALA =  ['Алматы', 'алматы', 'АЛМАТЫ', ' Алматы', 'Almaty']

print('Всего заказов в Алматы:', df['city'].isin(ALA).sum())
print('Отмененных заказов в Алматы:', (~paid & df['city'].isin(ALA)).sum())
print(f'Доля отменённых заказов в Алматы: {(~paid & df['city'].isin(ALA)).sum() / df['city'].isin(ALA).sum() * 100:.1f}%')

Всего заказов в Алматы: 2188
Отмененных заказов в Алматы: 172
Доля отменённых заказов в Алматы: 7.9%


In [14]:
print('Всего заказов в Шымкенте:', df['city'].isin(SH).sum())
print('Отмененных заказов в Шымкенте:', (~paid & df['city'].isin(SH)).sum())
print(f'Доля отменённых заказов в Шымкенте: {(~paid & df['city'].isin(SH)).sum() / df['city'].isin(SH).sum() * 100:.1f}%')

Всего заказов в Шымкенте: 804
Отмененных заказов в Шымкенте: 164
Доля отменённых заказов в Шымкенте: 20.4%


In [15]:
# для понимания картины с отменами по другим городам
AST = ['Астана', 'астана', 'Astana', 'Astana']
df1 = df.copy()

df1['city'] = df['city'].str.strip().replace(ALA, 'Алматы').replace(SH, 'Шымкент').replace(AST, 'Астана')
print(df1[~paid]['city'].value_counts())
print('')

print('Всего заказов в Астане:', df1['city'].isin(AST).sum())
print('Отмененных заказов в Астане:', (~paid & df1['city'].isin(AST)).sum())
print(f'Доля отменённых заказов в Астане: {(~paid & df1['city'].isin(AST)).sum() / df1['city'].isin(AST).sum() * 100:.1f}%')

city
Алматы       172
Шымкент      164
Астана       123
Караганда     48
Актобе        41
Атырау        11
Name: count, dtype: int64

Всего заказов в Астане: 1390
Отмененных заказов в Астане: 123
Доля отменённых заказов в Астане: 8.8%


Среди всех городов выделяются отмены в Шымкенте: 164 отмененных заказа, которые составляют 20,4%. Этот показатель в два раза выше доли отменных заказов во всех городах, и на 12,5% процентных пункта выше чем в Алматы.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [16]:
# 4
sh_sep = df[paid & not_test & sep & df['city'].isin(SH)]
sh_octo = df[paid & not_test & octo & df['city'].isin(SH)]
ala_sep = df[paid & not_test & sep & df['city'].isin(ALA)]
ala_octo = df[paid & not_test & octo & df['city'].isin(ALA)]

display(ala_octo.head(7))

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
2783,102694,2026-10-01 08:00,C01233,Алматы,Красота,8680.0,1.0,оплачен
2787,104436,2026-10-01 08:00,C02178,Алматы,Книги,6720.0,2.0,оплачен
2789,105224,2026-10-01 09:00,C02047,Алматы,Электроника,37900.0,1.0,оплачен
2790,104431,2026-10-01 09:00,C00249,Алматы,Одежда,4830.0,3.0,оплачен
2791,104194,2026-10-01 10:00,C00689,Алматы,Одежда,8250.0,3.0,оплачен
2797,104188,2026-10-01 12:00,C00690,АЛМАТЫ,Красота,5730.0,2.0,оплачен
2798,103751,2026-10-01 12:00,C00450,Алматы,Электроника,25060.0,1.0,оплачен


In [17]:
print('Выручка оплаченных заказов без TEST в Алматы за сентябрь:', ala_sep['amount'].sum())
print('Выручка оплаченных заказов без TEST в Алматы за октябрь:', ala_octo['amount'].sum())
print(f'Изменение (%): {round((ala_octo['amount'].sum() - ala_sep['amount'].sum()) / ala_sep['amount'].sum() * 100, 1)}%')

Выручка оплаченных заказов без TEST в Алматы за сентябрь: 15523940.0
Выручка оплаченных заказов без TEST в Алматы за октябрь: 14643020.0
Изменение (%): -5.7%


In [18]:
print('Выручка оплаченных заказов без TEST в Шымкенте за сентябрь:', sh_sep['amount'].sum())
print('Выручка оплаченных заказов без TEST в Шымкенте за октябрь:', sh_octo['amount'].sum())
print(f'Изменение (%): {round((sh_octo['amount'].sum() - sh_sep['amount'].sum()) / sh_sep['amount'].sum() * 100, 1)}%')

Выручка оплаченных заказов без TEST в Шымкенте за сентябрь: 4949800.0
Выручка оплаченных заказов без TEST в Шымкенте за октябрь: 4044010.0
Изменение (%): -18.3%


Падение сильнее в Шымкенте (-18,3%). Оно больше, чем в Алматы почти в 3 раза (в Алматы −5,7%), и составляет 12.6% процентных пункта. Следует разобраться в данных, связанные с Шымкентом, и понять что происходит, почему существует такая разница и проверить правильность полученных данных.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [19]:
# 5
cancel_sh = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]

display(cancel_sh.head(3))
print(cancel_sh.shape)

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
5,102231,2026-09-01 09:00,C00763,Шымкент,Книги,3670.0,1.0,отменён
19,100926,2026-09-01 11:00,C02131,Шымкент,Дом,12540.0,1.0,отменён
62,102059,2026-09-01 17:00,C00174,Шымкент,Дом,21080.0,3.0,отменён


(153, 8)


Первая ошибка в коде связана с тем, что для двух условий не даны скобки. В неправильном коде оператор `&` соеднияет не два условия, а `Шымкент` и `df['status']`. Если бы у нас были бы отдельные маски, то их можно было бы соединять без скобок, как это писали раньше (`df[not_test & paid]`). В нашем случае скобки помогают выполнить сначала условия, а затем соединить их через `&`.

Вторая ошибка связана с тем, что в коде учитывается только одно написание города Шымкент, без учета его неверных написаний (`шымкент`,  `Shymkent`). Для верного нахождения стоило или учитывать все вариации написаний, или исправить названия городов еще на этапе предобработки данных в самом начале работы. Итоговый верный код представлен ниже.

In [20]:
cancel_sh = df[((df["city"] == "Шымкент") | (df["city"] == "шымкент") |
                (df["city"] == "Shymkent")) & (df["status"] == "отменён")]
# либо через isin(SH), но такой код уже постоянно использовали в работе раньше
# для наглядности представил другое написание - более громоздкое, но тоже верное

display(cancel_sh.head(11))
print(cancel_sh.shape)

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
5,102231,2026-09-01 09:00,C00763,Шымкент,Книги,3670.0,1.0,отменён
19,100926,2026-09-01 11:00,C02131,Шымкент,Дом,12540.0,1.0,отменён
62,102059,2026-09-01 17:00,C00174,Шымкент,Дом,21080.0,3.0,отменён
116,100610,2026-09-02 10:00,C00101,Шымкент,Одежда,22450.0,2.0,отменён
174,100791,2026-09-02 17:00,C00188,Шымкент,Дом,13120.0,3.0,отменён
203,100226,2026-09-02 22:00,C01593,Шымкент,Электроника,36310.0,3.0,отменён
204,100226,2026-09-02 22:00,C01593,Шымкент,Электроника,36310.0,3.0,отменён
362,101421,2026-09-04 19:00,C00238,Шымкент,Книги,4100.0,3.0,отменён
435,100188,2026-09-05 16:00,C00483,Шымкент,Одежда,7780.0,3.0,отменён
509,102277,2026-09-06 13:00,C02159,Шымкент,Электроника,13230.0,3.0,отменён


(164, 8)


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [21]:
# 6
df['customer_id'].value_counts()[:5]

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64

Больше всего заказов сделал клиент `C01818`. Детально его заказы ниже.

In [22]:
best_customer = df['customer_id'] == 'C01818'
df.loc[best_customer, ['order_date', 'city', 'amount', 'status']]

,order_date,city,amount,status
57,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,2026-09-24 18:00,Астана,1490.0,оплачен
2617,2026-09-29 09:00,Астана,22170.0,оплачен
2776,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,2026-10-18 08:00,Астана,13600.0,оплачен
4610,2026-10-21 19:00,Алматы,21990.0,оплачен


In [23]:
print("Количество повторов по 'order_id' у самого частого клиента:", len(df[best_customer]) - df.loc[best_customer, 'order_id'].nunique())

Количество повторов по 'order_id' у самого частого клиента: 1


In [24]:
display(df[best_customer])

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
57,101125,2026-09-01 15:00,C01818,Шымкент,Дом,7350.0,1.0,оплачен
779,100096,2026-09-09 18:00,C01818,Шымкент,Электроника,45270.0,3.0,оплачен
1371,101504,2026-09-15 21:00,C01818,Алматы,Одежда,8890.0,1.0,оплачен
1375,101504,2026-09-15 21:00,C01818,Алматы,Одежда,8890.0,1.0,оплачен
2195,101860,2026-09-24 18:00,C01818,Астана,Красота,1490.0,1.0,оплачен
2617,101462,2026-09-29 09:00,C01818,Астана,Электроника,22170.0,2.0,оплачен
2776,101495,2026-09-30 21:00,C01818,Алматы,Электроника,20180.0,2.0,оплачен
3417,103416,2026-10-08 14:00,C01818,Караганда,Дом,4330.0,1.0,оплачен
4279,102644,2026-10-18 08:00,C01818,Астана,Дом,13600.0,3.0,оплачен
4610,104001,2026-10-21 19:00,C01818,Алматы,Одежда,21990.0,1.0,оплачен


Видим, что в нашей таблице у самого частого клиента есть заказы, который записаны по несколько раз. А это значит, что это, возможно, не единичный случай, и нам стоит убрать повторы из нашей таблицы для понимания реального положения вещей. Выручка, которую мы считали ранее, подсчитана неверно - она будет завышена из-за повторов строк. 

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [25]:
# 7 
not_test_and_paid['amount'].describe()

count      4901.000000
mean      14672.391349
std       13249.340787
min         720.000000
25%        5880.000000
50%       10300.000000
75%       18690.000000
max      139040.000000
Name: amount, dtype: float64

In [26]:
q3 = not_test_and_paid['amount'].quantile(0.75)
q1 = not_test_and_paid['amount'].quantile(0.25)

iqr = q3 - q1
upper = q3 + 1.5 * iqr
print('Верхняя граница:', upper)
print('Всего оплачено заказов без TEST:', len(not_test_and_paid))
print('Заказов выше границы:', len(not_test_and_paid[not_test_and_paid['amount'] > upper]))
print(f'Доля заказов выше границы: {len(not_test_and_paid[not_test_and_paid['amount'] > upper]) / 
                                    len(not_test_and_paid) * 100:.1f}%')

Верхняя граница: 37905.0
Всего оплачено заказов без TEST: 4901
Заказов выше границы: 320
Доля заказов выше границы: 6.5%


In [27]:
not_test_and_paid[not_test_and_paid['amount'] > upper]['category'].value_counts()

category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64

Наиболее частая категория заказов выше границы - Электроника (308 заказа). 

По сравнению с z-оценкой, где выбросов было 87, правила дают разные числа. Это происходит потому что нет однозначного подхода какие именно заказы стоит считать аномальными (выше определенных значений), поэтому применяются два разных подхода. IQR основывается на 50% числах от выборки, которые находятся между Q1 и Q3, и отдельные большие значения за их границами мало влияют на IQR, а z-оценка учитывает среднюю и стандартное отклонение от средней. То есть, в z-оценке рассматриваются абсолютно все значения, а не только в промежутке от Q1 до Q3, и очень большие значения увеличивают среднюю и отклонение. Поэтому при z-score вверхняя граница оказалось выше, а выбросов получилось меньше.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [28]:
# 8
series1 = df[~paid]['city'].value_counts()
series2 = df['city'].value_counts()
print((series1 / series2).sort_values(ascending=False))

city
Shymkent     0.230769
Шымкент      0.215190
алматы       0.140625
астана       0.130952
Актобе       0.103275
Astana       0.100000
Караганда    0.089219
Астана       0.084829
Алматы       0.079917
Атырау       0.075342
Almaty       0.057971
 Алматы      0.045455
шымкент      0.037037
АЛМАТЫ       0.032258
Name: count, dtype: float64


In [29]:
print('Проверка деления по ярлыкам:')
print(series1['Shymkent'] / series2['Shymkent'])

Проверка деления по ярлыкам:
0.23076923076923078


In [30]:
print(series2[SH])

city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64


Нет, о городе нельзя судить по одному написанию, потому что это только часть заказов. Для правильной картины, нужно привести названия городов к одной форме и делать это в самом начале работы на этапе предобработки данных, и уже потом производить любые вычисления и исследовательский анализ данных.